In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [9]:
import pandas as pd
import numpy as np

# Train verisini yükle
train = pd.read_csv(
    "/content/drive/MyDrive/datasets/store_sales/train.csv",
    parse_dates=["date"]
)

# Günlük toplam satış
daily_sales = (
    train.groupby("date", as_index=False)["sales"]
    .sum()
    .sort_values("date")
    .reset_index(drop=True)
)

daily_sales.head()


,date,sales
0,2013-01-01,2511.618999
1,2013-01-02,496092.417944
2,2013-01-03,361461.231124
3,2013-01-04,354459.677093
4,2013-01-05,477350.121229


In [10]:
daily_sales.to_csv(
    "/content/drive/MyDrive/datasets/store_sales/daily_sales.csv",
    index=False
)


In [13]:
# Tarih feature'ları
daily_sales["year"] = daily_sales["date"].dt.year
daily_sales["month"] = daily_sales["date"].dt.month
daily_sales["day"] = daily_sales["date"].dt.day
daily_sales["dayofweek"] = daily_sales["date"].dt.dayofweek
daily_sales["weekofyear"] = daily_sales["date"].dt.isocalendar().week.astype(int)

daily_sales.head()

## dayofweek: 0 = Pazartesi, 6 = Pazar

,date,sales,year,month,day,dayofweek,weekofyear
0,2013-01-01,2511.618999,2013,1,1,1,1
1,2013-01-02,496092.417944,2013,1,2,2,1
2,2013-01-03,361461.231124,2013,1,3,3,1
3,2013-01-04,354459.677093,2013,1,4,4,1
4,2013-01-05,477350.121229,2013,1,5,5,1


In [14]:
# Tarihe göre sırala (emin olmak için)
daily_sales = daily_sales.sort_values("date")

# Train - Test ayrımı
train = daily_sales[daily_sales["date"] < "2017-01-01"]
test  = daily_sales[daily_sales["date"] >= "2017-01-01"]

print("Train shape:", train.shape)
print("Test shape :", test.shape)

print("\nTrain tarih aralığı:")
print(train["date"].min(), "→", train["date"].max())

print("\nTest tarih aralığı:")
print(test["date"].min(), "→", test["date"].max())


Train shape: (1457, 7)
Test shape : (227, 7)

Train tarih aralığı:
2013-01-01 00:00:00 → 2016-12-31 00:00:00

Test tarih aralığı:
2017-01-01 00:00:00 → 2017-08-15 00:00:00


In [16]:
# Naive forecast: son train günü
last_train_value = train.iloc[-1]["sales"]

test = test.copy()
test.loc[:, "naive_pred"] = last_train_value

window = 7
ma_value = train["sales"].rolling(window=window).mean().iloc[-1]

test.loc[:, "ma_pred"] = ma_value

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_true = test["sales"]
y_naive = test["naive_pred"]
y_ma = test["ma_pred"]

print("Naive MAE :", mean_absolute_error(y_true, y_naive))
print("Naive RMSE:", np.sqrt(mean_squared_error(y_true, y_naive)))

print("MA MAE    :", mean_absolute_error(y_true, y_ma))
print("MA RMSE   :", np.sqrt(mean_squared_error(y_true, y_ma)))


Naive MAE : 273791.5759241837
Naive RMSE: 310699.23605658766
MA MAE    : 221071.40737158756
MA RMSE   : 253498.18046198785


In [17]:
# Feature Engineering başlangıcı
train_fe = train.copy()
test_fe  = test.copy()


In [19]:
for lag in [7, 14, 28]:
    train_fe[f"lag_{lag}"] = train_fe["sales"].shift(lag)
    test_fe[f"lag_{lag}"]  = test_fe["sales"].shift(lag)

train_fe.head(10)


,date,sales,year,month,day,dayofweek,weekofyear,lag_7,lag_14,lag_28
0,2013-01-01,2511.618999,2013,1,1,1,1,NaN,NaN,NaN
1,2013-01-02,496092.417944,2013,1,2,2,1,NaN,NaN,NaN
2,2013-01-03,361461.231124,2013,1,3,3,1,NaN,NaN,NaN
3,2013-01-04,354459.677093,2013,1,4,4,1,NaN,NaN,NaN
4,2013-01-05,477350.121229,2013,1,5,5,1,NaN,NaN,NaN
5,2013-01-06,519695.401088,2013,1,6,6,1,NaN,NaN,NaN
6,2013-01-07,336122.801066,2013,1,7,0,2,NaN,NaN,NaN
7,2013-01-08,318347.777981,2013,1,8,1,2,2511.618999,NaN,NaN
8,2013-01-09,302530.809018,2013,1,9,2,2,496092.417944,NaN,NaN
9,2013-01-10,258982.003049,2013,1,10,3,2,361461.231124,NaN,NaN


In [20]:
for window in [7, 14, 28]:
    train_fe[f"roll_mean_{window}"] = (
        train_fe["sales"]
        .shift(1)
        .rolling(window=window)
        .mean()
    )

    test_fe[f"roll_mean_{window}"] = (
        test_fe["sales"]
        .shift(1)
        .rolling(window=window)
        .mean()
    )
train_fe.head(15)


,date,sales,year,month,day,dayofweek,weekofyear,lag_7,lag_14,lag_28,roll_mean_7,roll_mean_14,roll_mean_28
0,2013-01-01,2511.618999,2013,1,1,1,1,NaN,NaN,NaN,NaN,NaN,NaN
1,2013-01-02,496092.417944,2013,1,2,2,1,NaN,NaN,NaN,NaN,NaN,NaN
2,2013-01-03,361461.231124,2013,1,3,3,1,NaN,NaN,NaN,NaN,NaN,NaN
3,2013-01-04,354459.677093,2013,1,4,4,1,NaN,NaN,NaN,NaN,NaN,NaN
4,2013-01-05,477350.121229,2013,1,5,5,1,NaN,NaN,NaN,NaN,NaN,NaN
5,2013-01-06,519695.401088,2013,1,6,6,1,NaN,NaN,NaN,NaN,NaN,NaN
6,2013-01-07,336122.801066,2013,1,7,0,2,NaN,NaN,NaN,NaN,NaN,NaN
7,2013-01-08,318347.777981,2013,1,8,1,2,2511.618999,NaN,NaN,363956.181220,NaN,NaN
8,2013-01-09,302530.809018,2013,1,9,2,2,496092.417944,NaN,NaN,409075.632504,NaN,NaN
9,2013-01-10,258982.003049,2013,1,10,3,2,361461.231124,NaN,NaN,381423.974086,NaN,NaN


In [21]:
train_fe = train_fe.dropna().reset_index(drop=True)
test_fe  = test_fe.dropna().reset_index(drop=True)

train_fe.isna().sum()


,0
date,0
sales,0
year,0
month,0
day,0
dayofweek,0
weekofyear,0
lag_7,0
lag_14,0
lag_28,0


In [22]:
FEATURES = [
    "year", "month", "day", "dayofweek", "weekofyear",
    "lag_7", "lag_14", "lag_28",
    "roll_mean_7", "roll_mean_14", "roll_mean_28"
]

X_train = train_fe[FEATURES]
y_train = train_fe["sales"]

X_test  = test_fe[FEATURES]
y_test  = test_fe["sales"]

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)


(1429, 11) (1429,)
(199, 11) (199,)


In [23]:
!pip install lightgbm


In [24]:
import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=7,
    random_state=42
)

lgb_model.fit(X_train, y_train)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000970 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1641
[LightGBM] [Info] Number of data points in the train set: 1429, number of used features: 11
[LightGBM] [Info] Start training from score 608759.316807
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

LGBMRegressor(learning_rate=0.05, max_depth=7, n_estimators=500,
              random_state=42)

In [26]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_pred = lgb_model.predict(X_test)

mae_lgb = mean_absolute_error(y_test, y_pred)
rmse_lgb = np.sqrt(mean_squared_error(y_test, y_pred))

print("LightGBM MAE :", mae_lgb)
print("LightGBM RMSE:", rmse_lgb)


LightGBM MAE : 81514.11153068337
LightGBM RMSE: 107323.13789485004


## Model Performans Değerlendirmesi

Bu aşamada oluşturulan LightGBM modeli, klasik zaman serisi yöntemleriyle
karşılaştırılarak değerlendirilmiştir.

Naive ve Hareketli Ortalama (Moving Average) yaklaşımları, satış verisinin
temel trendini yakalayabilse de ani dalgalanmalar ve mevsimsellik karşısında
yetersiz kalmıştır.

LightGBM modeli ise gecikmeli değişkenler (lag features) ve hareketli ortalama
özellikleri sayesinde geçmiş satış davranışını daha iyi öğrenmiş ve tahmin
hatalarında belirgin bir düşüş sağlamıştır.

Elde edilen sonuçlar, makine öğrenmesi tabanlı modellerin zaman serisi
problemlerinde, özellikle uygun feature engineering uygulandığında,
klasik yöntemlere kıyasla daha yüksek doğruluk sunduğunu göstermektedir.
